In [1]:
import duckdb

con = duckdb.connect("../telco.duckdb")

with open("../sql/01_schema.sql") as f:
    con.execute(f.read())

# Confirm the four tables exist
con.execute("SELECT table_name FROM information_schema.tables ORDER BY 1").df()

,table_name
0,billing
1,customer
2,geography
3,subscription


In [ ]:
con.execute("DESCRIBE customer").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,zip_code,INTEGER,NO,NaN,None,None
2,gender,VARCHAR,NO,NaN,None,None
3,senior_citizen,VARCHAR,NO,NaN,None,None
4,partner,VARCHAR,NO,NaN,None,None
5,dependents,VARCHAR,NO,NaN,None,None
6,tenure_months,INTEGER,NO,NaN,None,None
7,churn_label,VARCHAR,NO,NaN,None,None
8,churn_value,INTEGER,NO,NaN,None,None
9,churn_reason,VARCHAR,YES,NaN,None,None


In [3]:
con.execute("DESCRIBE geography").df()

,column_name,column_type,null,key,default,extra
0,zip_code,INTEGER,NO,PRI,None,None
1,city,VARCHAR,NO,NaN,None,None
2,latitude,DOUBLE,NO,NaN,None,None
3,longitude,DOUBLE,NO,NaN,None,None


In [4]:
con.execute("DESCRIBE billing").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,paperless_billing,VARCHAR,NO,NaN,None,None
2,payment_method,VARCHAR,NO,NaN,None,None
3,monthly_charges,"DECIMAL(10,2)",NO,NaN,None,None
4,total_charges,"DECIMAL(10,2)",NO,NaN,None,None


In [5]:
con.execute("DESCRIBE subscription").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,phone_service,VARCHAR,NO,NaN,None,None
2,multiple_lines,VARCHAR,NO,NaN,None,None
3,internet_service,VARCHAR,NO,NaN,None,None
4,online_security,VARCHAR,NO,NaN,None,None
5,online_backup,VARCHAR,NO,NaN,None,None
6,device_protection,VARCHAR,NO,NaN,None,None
7,tech_support,VARCHAR,NO,NaN,None,None
8,streaming_tv,VARCHAR,NO,NaN,None,None
9,streaming_movies,VARCHAR,NO,NaN,None,None


In [6]:
import pandas as pd, kagglehub

path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
raw = pd.read_excel(f"{path}/Telco_customer_churn.xlsx")

con.register("raw", raw)

c:\Users\IC Clearwater\OneDrive\Documents\GitHub\Customer_Lifetime_Value_Churn_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
con.execute("SELECT COUNT(*) FROM raw").df()

,count_star()
0,7043


In [8]:
with open("../sql/02_load.sql") as f:
    con.execute(f.read())

con.execute("SELECT COUNT(*) FROM geography").df()

,count_star()
0,1652


In [9]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM customer").df()

,count_star()
0,7043


In [14]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM subscription").df()

,count_star()
0,7043


In [11]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM billing").df()

,count_star()
0,7043


In [15]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("""
    SELECT 'geography' AS t, COUNT(*) FROM geography
    UNION ALL SELECT 'customer', COUNT(*) FROM customer
    UNION ALL SELECT 'subscription', COUNT(*) FROM subscription
    UNION ALL SELECT 'billing', COUNT(*) FROM billing
""").df()

,t,count_star()
0,geography,1652
1,customer,7043
2,subscription,7043
3,billing,7043


In [16]:
con.execute("""
    SELECT 'churned customers'        AS check, COUNT(*) AS value FROM customer WHERE churn_value = 1
    UNION ALL
    SELECT 'churn_reason populated',   COUNT(churn_reason) FROM customer
    UNION ALL
    SELECT 'zero total_charges',       COUNT(*) FROM billing WHERE total_charges = 0
    UNION ALL
    SELECT 'zero tenure',              COUNT(*) FROM customer WHERE tenure_months = 0
    UNION ALL
    SELECT 'full four-table join',     COUNT(*)
        FROM customer c
        JOIN geography g    ON c.zip_code    = g.zip_code
        JOIN subscription s ON c.customer_id = s.customer_id
        JOIN billing b      ON c.customer_id = b.customer_id
""").df()

,check,value
0,churned customers,1869
1,churn_reason populated,1869
2,zero total_charges,11
3,zero tenure,11
4,full four-table join,7043
